# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset overview
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined in the Croissant schema.

In [ ]:
# List available record sets by @id and display their fields
print("Available record sets and their fields:\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', 'N/A')}")
    print("")
# Save record set @ids for extraction later
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all records in each record set into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, print the first record set's id, column names, and preview
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in schema.")

## 4. Exploratory Data Analysis (EDA)
We now explore the content of a selected record set. We'll select numeric fields by their `@id`, filter rows based on a threshold, normalize values, and optionally group by categorical fields. All references are made using `@id`.

In [ ]:
import numpy as np

# Choose the first record set as the main table for EDA if available
if record_set_ids:
    rs_id = main_record_set_id
    df = dataframes[rs_id]
    
    # Try to find a numeric field by its @id
    try:
        fields = next(rs for rs in dataset.record_sets if rs.id == rs_id).fields
        # Select first float/integer field
        numeric_field = None
        for f in fields:
            # Data type from schema is at f.data_type
            if hasattr(f, 'data_type') and f.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field = f.id
                break
        if numeric_field and numeric_field in df.columns:
            # Convert to numeric
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            threshold = df[numeric_field].mean() if df[numeric_field].notna().sum() > 0 else 0
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())
            # Normalize
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
                filtered_df[numeric_field].std()
            )
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a categorical field
            group_field = None
            for f in fields:
                if hasattr(f, 'data_type') and f.data_type == 'schema:Text' and f.id != numeric_field:
                    group_field = f.id
                    break
            if group_field and group_field in filtered_df.columns:
                grouped_df = (
                    filtered_df.groupby(group_field)[numeric_field].mean().to_frame("mean_")
                )
                print(f"Grouped mean of {numeric_field} by {group_field}:")
                display(grouped_df.head())
            else:
                print("No suitable categorical field for grouping found.")
        else:
            print("No suitable numeric field found in this record set.")
    except Exception as e:
        print(f"Could not perform EDA: {e}")
else:
    print("No record sets to analyze.")

## 5. Visualization
Visualize distributions or relationships between fields in the main record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {rs_id}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    # If grouping, make a boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we:

- Loaded dataset metadata and explored record sets using `mlcroissant`.
- Extracted records from the main record set and identified fields by their `@id`.
- Performed basic exploratory data analysis such as filtering, normalization, and optional grouping by categorical fields (all referencing Croissant `@id`).
- Visualized data characteristics to assist further research.

**Tip:** The FAIR^2 dataset is structured with rich metadata; reviewing schema documentation will help you interpret each field's meaning. Customize the EDA section for specific analytical or research needs!